# PhishGuard AI — Member 2 (Nishen Madawa, CIT-24-01-0256)
## Logistic Regression (ML) + GRU (DL)

**Branch:** `feature/nishen_CIT-24-01-0256`
**Feature engineering:** TF-IDF (for LR) + Keras tokenizer & padded sequences (for GRU)

Run the cells **top to bottom**. Colab → Runtime → Change runtime type → **T4 GPU** before you start
(the GRU section needs it, otherwise training takes hours).

## 1. Setup — libraries and reproducibility

In [ ]:
# Colab already has most of these; this just makes sure.
!pip install -q nltk scikit-learn tensorflow pandas numpy matplotlib seaborn tabulate

import os, re, string, pickle, random, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Embedding, GRU, Dropout, Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Fixed seeds so my results are reproducible (viva question: "can you reproduce this?")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))
print("All libraries loaded")

In [ ]:
# Mount Google Drive so models/screenshots survive between Colab sessions
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/PhishGuard_Member2'
for folder in ['data', 'models', 'screenshots', 'notebooks']:
    os.makedirs(f'{BASE}/{folder}', exist_ok=True)

print("Working folder ready at:", BASE)

## 2. Data collection

**Stage 1 of the pipeline.** The Kaggle download gives you *seven* CSV files
(Enron, Ling, Nazario, CEAS_08, Nigerian_Fraud, SpamAssasin, and the combined
`phishing_email.csv`). Only `phishing_email.csv` is the full ~82,500-email
dataset described in the project brief — the others are its individual sources.

In [ ]:
# Download via Kaggle API.
# 1. kaggle.com -> profile -> Settings -> API -> Create New API Token (downloads kaggle.json)
# 2. Run this cell and upload that kaggle.json when prompted.
from google.colab import files
uploaded = files.upload()          # choose kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!pip install -q --upgrade kaggle
!kaggle datasets download -d naserabdullahalam/phishing-email-dataset -p /content/data --unzip

print()
print("Downloaded files:")
!ls -lh /content/data

In [ ]:
# ---- Pick the RIGHT csv ----------------------------------------------------
# Do NOT just take the first file the glob finds: alphabetically that is CEAS_08.csv
# or Ling.csv, which are small single-source subsets (~2-40k rows) and give
# misleadingly easy results. We explicitly want the combined file.
TARGET = 'phishing_email.csv'

candidates = glob.glob('/content/data/**/*.csv', recursive=True) + glob.glob('/content/*.csv')
candidates = sorted(set(candidates))
print("CSV files found:")
for c in candidates:
    print("  ", c)

matches = [c for c in candidates if os.path.basename(c).lower() == TARGET]
if not matches:
    raise FileNotFoundError(
        f"Could not find {TARGET}. Check the list above and set DATA_PATH manually.")

DATA_PATH = matches[0]
print("\nUsing dataset:", DATA_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
df.head()

In [ ]:
# ---- Normalise column names to 'text' and 'label' --------------------------
# phishing_email.csv uses 'text_combined'; the source files use 'subject' + 'body'.
# This handles both so the notebook does not silently break if you switch file.
if 'text_combined' in df.columns:
    df = df.rename(columns={'text_combined': 'text'})
elif 'body' in df.columns:
    if 'subject' in df.columns:
        df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
    else:
        df = df.rename(columns={'body': 'text'})
elif 'Email Text' in df.columns:
    df = df.rename(columns={'Email Text': 'text'})

if 'text' not in df.columns:
    raise KeyError(f"No text column found. Columns are: {df.columns.tolist()}")

df = df[['text', 'label']].copy()
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['text', 'label'])
df['label'] = df['label'].astype(int)
df = df[df['text'].astype(str).str.strip() != ''].reset_index(drop=True)

print("Shape after cleaning missing values:", df.shape)
print()
print("Class counts:")
print(df['label'].value_counts())
print()
print("Class balance (%):")
print((df['label'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Read a couple of real emails — Week 2 of the task guide asks for this
print("--- SAMPLE PHISHING (label = 1) ---")
print(df[df['label'] == 1]['text'].iloc[0][:600])
print()
print("--- SAMPLE LEGITIMATE (label = 0) ---")
print(df[df['label'] == 0]['text'].iloc[0][:600])

In [ ]:
# ---- Optional sub-sample ---------------------------------------------------
# Keep SAMPLE_SIZE = None to use the FULL dataset (what the brief asks for).
# Only set a number if Colab keeps disconnecting; if you do, say so in your report.
SAMPLE_SIZE = None

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(df, train_size=SAMPLE_SIZE,
                             stratify=df['label'], random_state=SEED)
    df = df.reset_index(drop=True)
    print(f"NOTE: using a stratified sample of {SAMPLE_SIZE} emails")

print("Working dataset shape:", df.shape)
print(df['label'].value_counts())

## 3. Preprocessing (Stage 2)

Lowercase → strip URLs/emails/numbers/punctuation → tokenize (NLTK) →
remove stopwords → lemmatize. This is the exact list to recite in the viva.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
_punct_table = str.maketrans('', '', string.punctuation)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)          # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)                 # remove email addresses
    text = text.translate(_punct_table)                    # remove punctuation
    text = re.sub(r'\d+', ' ', text)                      # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

print("Before:", df['text'].iloc[0][:300])
print()
print("After :", clean_text(df['text'].iloc[0][:300]))

In [ ]:
# Apply to the FULL dataset. On ~82,500 emails this takes roughly 5-12 minutes
# in Colab — that is normal, let it finish.
import time
t0 = time.time()
df['cleaned_text'] = df['text'].apply(clean_text)
print(f"Preprocessing took {time.time() - t0:.1f}s")

# Drop rows that became empty after cleaning (e.g. emails that were only URLs)
before = len(df)
df = df[df['cleaned_text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows that were empty after cleaning")
print("Final shape after preprocessing:", df.shape)
df[['text', 'cleaned_text', 'label']].head()

In [ ]:
# Checkpoint so you never have to redo preprocessing
df.to_csv(f'{BASE}/data/cleaned_dataset_member2.csv', index=False)
print("Saved cleaned dataset to Drive")

# To resume in a later session, uncomment:
# df = pd.read_csv(f'{BASE}/data/cleaned_dataset_member2.csv').dropna(subset=['cleaned_text'])

## 4. EDA (Stage 3) — two required charts

In [ ]:
# --- Chart 1: class distribution ---
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, hue='label',
              palette=['#2ecc71', '#e74c3c'], legend=False)
plt.xticks([0, 1], ['Legitimate (0)', 'Phishing (1)'])
plt.title('Class Distribution — Phishing vs Legitimate')
plt.xlabel('')
plt.ylabel('Number of Emails')
for i, v in enumerate(df['label'].value_counts().sort_index()):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_class_dist.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 2: top 50 words in phishing emails ---
from collections import Counter

phishing_words = ' '.join(df[df['label'] == 1]['cleaned_text']).split()
top_50 = Counter(phishing_words).most_common(50)
words, counts = zip(*top_50)

plt.figure(figsize=(10, 12))
sns.barplot(x=list(counts), y=list(words), hue=list(words),
            palette='Reds_r', legend=False)
plt.title('Top 50 Most Frequent Words — Phishing Emails')
plt.xlabel('Frequency')
plt.ylabel('')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_top_words.png', dpi=150)
plt.show()

## 5. Feature engineering (Stage 4) — TF-IDF, 80/20 split

The split happens **before** the vectorizer is fitted. Fitting TF-IDF on all the
data first would leak test-set vocabulary into training and inflate the scores.

In [ ]:
X = df['cleaned_text']
y = df['label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print("Train emails:", len(X_train_text), " Test emails:", len(X_test_text))

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train_text)   # fit on TRAIN only
X_test_tfidf  = tfidf.transform(X_test_text)        # transform test

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test  TF-IDF shape:", X_test_tfidf.shape)

with open(f'{BASE}/models/tfidf_member2.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer saved -> models/tfidf_member2.pkl")

## 6. ML model — Logistic Regression (Stage 5)

In [ ]:
lr_baseline = LogisticRegression(max_iter=1000, random_state=SEED)
lr_baseline.fit(X_train_tfidf, y_train)
y_pred_base = lr_baseline.predict(X_test_tfidf)

print("=== Logistic Regression — BASELINE (C = 1.0, default) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_base, zero_division=0):.4f}")
print()
print(classification_report(y_test, y_pred_base, zero_division=0,
                            target_names=['Legitimate', 'Phishing']))

### 6.1 Hyperparameter tuning — the important fix

Picking `C` by looking at the **test** score is a methodology error: the test set
stops being unseen data and every number after it is optimistic. `GridSearchCV`
below tunes `C` using cross-validation **inside the training set only**, and the
test set is touched exactly once, at the end.

In [ ]:
param_grid = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced'),
    param_grid, scoring='f1', cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train_tfidf, y_train)          # TRAIN data only

print("\nCV F1 for each C (training data only):")
for C, mean, std in zip(grid.cv_results_['param_C'],
                        grid.cv_results_['mean_test_score'],
                        grid.cv_results_['std_test_score']):
    print(f"  C={C:<7} F1 = {mean:.4f} +/- {std:.4f}")

best_C = grid.best_params_['C']
print(f"\nBest C = {best_C}  (CV F1 = {grid.best_score_:.4f})")

In [ ]:
# 10-fold cross-validation on the training set with the chosen C
lr_final = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED,
                              class_weight='balanced')

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(lr_final, X_train_tfidf, y_train, cv=cv10,
                            scoring='f1', n_jobs=-1)
print("10-fold CV F1 scores:", np.round(cv_scores, 4))
print(f"Mean CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

# Fit on the whole training set, then evaluate ONCE on the held-out test set
lr_final.fit(X_train_tfidf, y_train)
y_pred_lr  = lr_final.predict(X_test_tfidf)
y_proba_lr = lr_final.predict_proba(X_test_tfidf)[:, 1]

lr_accuracy  = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, zero_division=0)
lr_recall    = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1        = f1_score(y_test, y_pred_lr, zero_division=0)
lr_roc_auc   = roc_auc_score(y_test, y_proba_lr)

print()
print("=== Logistic Regression — FINAL (held-out test set) ===")
print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1-score : {lr_f1:.4f}")
print(f"ROC-AUC  : {lr_roc_auc:.4f}")
print()
print(classification_report(y_test, y_pred_lr, zero_division=0,
                            target_names=['Legitimate', 'Phishing']))

In [ ]:
# Confusion matrix — know these four numbers for the viva
cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel()
print(f"True Negatives : {tn}   (legitimate, correctly allowed)")
print(f"False Positives: {fp}   (legitimate wrongly flagged as phishing)")
print(f"False Negatives: {fn}   (phishing that slipped through - the dangerous one)")
print(f"True Positives : {tp}   (phishing, correctly caught)")

disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                              display_labels=['Legitimate', 'Phishing'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_lr_cm.png', dpi=150)
plt.show()

In [ ]:
# ROC curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
plt.figure(figsize=(6, 5))
plt.plot(fpr_lr, tpr_lr, color='#e74c3c',
         label=f'Logistic Regression (AUC = {lr_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_lr_roc.png', dpi=150)
plt.show()

In [ ]:
# Most influential features — good viva material ("which words trigger it?")
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_final.coef_[0]
top_phish = feature_names[np.argsort(coefs)[-20:][::-1]]
top_legit = feature_names[np.argsort(coefs)[:20]]
print("Top 20 words pushing towards PHISHING:")
print(', '.join(top_phish))
print()
print("Top 20 words pushing towards LEGITIMATE:")
print(', '.join(top_legit))

In [ ]:
with open(f'{BASE}/models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_final, f)
print("Final tuned Logistic Regression saved -> models/lr_model.pkl")

## 7. DL model — GRU (Stages 5-6)

**Architecture:** Embedding(128) → GRU(128) → Dropout(0.3) → Dense(64, ReLU) → Dense(1, Sigmoid)

A GRU is a gated recurrent network: it reads the email word by word and uses an
update gate and a reset gate to decide how much earlier context to keep. It has
two gates instead of the LSTM's three, so it trains faster with similar accuracy —
that is the answer to "why GRU and not LSTM?"

The Keras tokenizer is fitted on the **training text only**, same reason as TF-IDF.

In [ ]:
MAX_WORDS = 20000     # vocabulary size
MAX_LEN   = 200       # sequence length required by the brief
EMBED_DIM = 128

keras_tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
keras_tok.fit_on_texts(X_train_text)          # TRAIN only

X_train_seq = pad_sequences(keras_tok.texts_to_sequences(X_train_text),
                            maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(keras_tok.texts_to_sequences(X_test_text),
                            maxlen=MAX_LEN, padding='post', truncating='post')

y_train_arr = np.asarray(y_train, dtype='float32')
y_test_arr  = np.asarray(y_test,  dtype='float32')

vocab_size = min(MAX_WORDS, len(keras_tok.word_index) + 1)
print("Vocabulary size:", vocab_size)
print("X_train_seq:", X_train_seq.shape, " X_test_seq:", X_test_seq.shape)

with open(f'{BASE}/models/keras_tokenizer_member2.pkl', 'wb') as f:
    pickle.dump(keras_tok, f)
print("Keras tokenizer saved -> models/keras_tokenizer_member2.pkl")

In [ ]:
# Build the GRU. Using an explicit Input layer keeps this working on Keras 3
# (the old `input_length=` argument to Embedding was removed there).
gru_model = Sequential([
    Input(shape=(MAX_LEN,), dtype='int32', name='tokens'),
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, name='embedding'),
    GRU(128, name='gru'),
    Dropout(0.3, name='dropout'),
    Dense(64, activation='relu', name='dense_hidden'),
    Dense(1, activation='sigmoid', name='output'),
], name='PhishGuard_GRU_Member2')

gru_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

gru_model.summary()

In [ ]:
# Handle class imbalance so the GRU does not just predict the majority class
classes = np.unique(y_train_arr)
weights = compute_class_weight('balanced', classes=classes, y=y_train_arr)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weights)

CKPT = f'{BASE}/models/gru_model.keras'      # Keras 3 requires the .keras extension

callbacks = [
    ModelCheckpoint(CKPT, monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=3,
                  restore_best_weights=True, verbose=1),
]

history = gru_model.fit(
    X_train_seq, y_train_arr,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1)

print("\nTraining finished. Best model saved to:", CKPT)

In [ ]:
# Training vs validation loss curve (required deliverable)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history['loss'], label='Training loss')
axes[0].plot(history.history['val_loss'], label='Validation loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('GRU — Training vs Validation Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Training accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('GRU — Training vs Validation Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_gru_loss.png', dpi=150)
plt.show()

In [ ]:
# Evaluate the BEST checkpoint (not necessarily the last epoch)
best_gru = load_model(CKPT)

y_proba_gru = best_gru.predict(X_test_seq, batch_size=256, verbose=1).ravel()
y_pred_gru  = (y_proba_gru >= 0.5).astype(int)

gru_accuracy  = accuracy_score(y_test_arr, y_pred_gru)
gru_precision = precision_score(y_test_arr, y_pred_gru, zero_division=0)
gru_recall    = recall_score(y_test_arr, y_pred_gru, zero_division=0)
gru_f1        = f1_score(y_test_arr, y_pred_gru, zero_division=0)
gru_roc_auc   = roc_auc_score(y_test_arr, y_proba_gru)

print()
print("=== GRU — FINAL (held-out test set) ===")
print(f"Accuracy : {gru_accuracy:.4f}")
print(f"Precision: {gru_precision:.4f}")
print(f"Recall   : {gru_recall:.4f}")
print(f"F1-score : {gru_f1:.4f}")
print(f"ROC-AUC  : {gru_roc_auc:.4f}")
print()
print(classification_report(y_test_arr, y_pred_gru, zero_division=0,
                            target_names=['Legitimate', 'Phishing']))

In [ ]:
cm_gru = confusion_matrix(y_test_arr, y_pred_gru)
tn_g, fp_g, fn_g, tp_g = cm_gru.ravel()
print(f"True Negatives : {tn_g}")
print(f"False Positives: {fp_g}")
print(f"False Negatives: {fn_g}")
print(f"True Positives : {tp_g}")

disp = ConfusionMatrixDisplay(confusion_matrix=cm_gru,
                              display_labels=['Legitimate', 'Phishing'])
disp.plot(cmap='Purples', values_format='d')
plt.title('GRU — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_gru_cm.png', dpi=150)
plt.show()

In [ ]:
fpr_gru, tpr_gru, _ = roc_curve(y_test_arr, y_proba_gru)

plt.figure(figsize=(6, 5))
plt.plot(fpr_gru, tpr_gru, color='#8e44ad', label=f'GRU (AUC = {gru_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('GRU — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_gru_roc.png', dpi=150)
plt.show()

# Both models on one axis — useful for the report
plt.figure(figsize=(6, 5))
plt.plot(fpr_lr,  tpr_lr,  color='#e74c3c', label=f'Logistic Regression (AUC = {lr_roc_auc:.4f})')
plt.plot(fpr_gru, tpr_gru, color='#8e44ad', label=f'GRU (AUC = {gru_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Member 2 — ROC Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{BASE}/screenshots/m2_roc_comparison.png', dpi=150)
plt.show()

## 8. Results table (Stage 7) — this is what you send to Shakkya

In [ ]:
results = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_accuracy, 'Precision': lr_precision,
     'Recall': lr_recall, 'F1': lr_f1, 'ROC-AUC': lr_roc_auc},
    {'Model': 'GRU',                 'Accuracy': gru_accuracy, 'Precision': gru_precision,
     'Recall': gru_recall, 'F1': gru_f1, 'ROC-AUC': gru_roc_auc},
]).round(4)

results.to_csv(f'{BASE}/models/m2_results.csv', index=False)

print("Member 2 (Nishen Madawa, CIT-24-01-0256) — Final Results")
print(f"Dataset: {os.path.basename(DATA_PATH)}  |  {len(df):,} emails  |  80/20 split\n")
print(results.to_markdown(index=False))
results

In [ ]:
# Copy everything into the cloned repo folders so the file names match the
# deliverable list in the task guide, then you can commit them.
REPO = '/content/NLP_Group_16'      # change if you cloned somewhere else

if os.path.isdir(REPO):
    import shutil
    for sub in ['models', 'screenshots', 'notebooks']:
        os.makedirs(f'{REPO}/{sub}', exist_ok=True)
    for src in glob.glob(f'{BASE}/models/*') + glob.glob(f'{BASE}/screenshots/*'):
        sub = 'models' if '/models/' in src else 'screenshots'
        shutil.copy2(src, f'{REPO}/{sub}/{os.path.basename(src)}')
    print("Copied deliverables into", REPO)
    !ls -lh {REPO}/models {REPO}/screenshots
else:
    print(f"{REPO} not found — clone the repo there first, or download the files "
          f"from Drive at {BASE} and add them manually.")

## 9. Written analysis (for Shakkya / the report)

> Fill in the bracketed numbers from the results table above.

Logistic Regression on TF-IDF features reached an F1 of **[LR F1]** with an ROC-AUC of
**[LR AUC]**. Its strength is interpretability — the model coefficients directly show
which terms drive a phishing prediction, which supports the explanation feature of the
app — and it trains in seconds on the full dataset. Its weakness is that TF-IDF treats
the email as a bag of words, so word order and phrasing are lost.

The GRU reached an F1 of **[GRU F1]** and an ROC-AUC of **[GRU AUC]**. Because it reads
the email as a sequence, it captures phrasing such as "your account will be suspended"
as a pattern rather than three independent words. Its weaknesses are cost (it needs a
GPU and several minutes per epoch) and opacity — there is no coefficient to point at
when explaining a decision.

For this dataset the two models perform [similarly / LR ahead / GRU ahead], which
suggests that phishing emails are largely identifiable from vocabulary alone; the
sequential context the GRU adds gives [a small / no meaningful] extra gain for a
substantially higher training cost.